# 1.1 Calculate temperature from MI data using Thermobar

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🌡️ &nbsp; Thermobar </b> is an open-source python3 tool for performing mineral and mineral-melt thermobarometry, hygrometry and chemometry.

More information on Thermobar can be found at https://thermobar.readthedocs.io/en/latest/
</div>

A temperature is required to calculate pressures and fluid compositions from MI data. 

## 1.1.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import Thermobar as tb

import os
os.makedirs("output", exist_ok=True) # creates the output folder

When reporting calculations in manuscripts, it's important to know which version of the Python package the results you are showing used - so we can output those below. This notebook was created using Thermobar: 1.0.72

In [ ]:
print(f"\nThermobar: {tb.__version__}")

## 1.1.2 Import data

Typically, you'll have a spreadsheet full of MI and/or FI analyses to process. In this case, we've created a simplified xlsx file derived from the spreadsheets provided in Supplementary Materials of the original papers (i.e., some columns and rows have been deleted to make them easier to use). MI and MG data are from Wieser et al. (2021), whilst FI data is from DeVitre & Wieser (2024), which we will use in future notebooks.

In [ ]:
df_MI = pd.read_excel("input/Kilauea_MI_FI_MG_data.xlsx", sheet_name='MI_Data') # creates a DataFrame of just data in the MI_Data sheet of the excel spreadsheet
df_MG = pd.read_excel("input/Kilauea_MI_FI_MG_data.xlsx", sheet_name='MG_data') # ... as above for the MG_Data sheet

We can plot the data to see how it looks - like the H<sub>2</sub>O and CO<sub>2</sub> content of melt inclusions and matrix glass.

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4,4)) # creates a one panel figure using matplotlib

# MI: open circles - labels it MI for the legend
ax1.plot(df_MI['H2O'], df_MI['CO2_ppm'], 'ok', mfc='white', label = "MI")
# MG: x symbols - labels it MG for the legend
ax1.plot(df_MG['H2O'], df_MG['CO2_ppm'], 'xk', label="MG")

ax1.set_xlabel('H$_2$O [m] (wt%)') # labels the x-axis
ax1.set_ylabel('CO$_2$ [m] (ppm)') # labels the y-axis
ax1.legend(frameon=False) # adds a legend

H<sub>2</sub>O contents of melt inclusions commonly show signs of "resetting", wherein dissolved H<sub>2</sub>O concentrations are <em>lower</em> than those at the time of entrapment due to diffusive requilibration throught the crystal host during crustal magma storage (Wieser et al. 2021). For this exercise, we set the MI H<sub>2</sub>O contents to 0.5 wt% as in Wieser et al. (2021).

In [ ]:
for row in range(0,len(df_MI),1):
    df_MI.loc[row,'H2O'] = 0.5

## 1.1.3 Exploring options

Thermobar has a huge variety of thermometers available to choose from. Run the next cell to see the options available in Thermobar just for liquid-only thermometers.

In [ ]:
help(tb.calculate_liq_only_temp)

Following Wieser et al. (2021), we calculate the temperature using the MgO-liquid thermometer of Helz & Thornber (1987) implemented in Thermobar. 

## 1.1.4 Run calculations

Thermobar requires specific column names to recognise what each column contains - so first we'll need to rename the columns to be compatible with Thermobar. Thermobar also assumes the composition is in wt%, so anything that isn't would need to be converted. Then we can run the calculation and add the results to the original dataframe!

In [ ]:
# renames column names to be compatible with Thermobar
MI_tb=df_MI.copy()
MI_tb = MI_tb.rename(columns = {"SiO2":"SiO2_Liq","TiO2":"TiO2_Liq","Al2O3":"Al2O3_Liq", 
                             "MnO":"MnO_Liq","MgO":"MgO_Liq","CaO":"CaO_Liq",
                             "Na2O":"Na2O_Liq","K2O":"K2O_Liq","P2O5":"P2O5_Liq","H2O":"H2O_Liq"})

# and add a FeOt column
MI_tb['FeOt_Liq']=MI_tb['FeO']+MI_tb['Fe2O3']*0.8998
                             
# calculate temperature in celcius (hence -273.15) using MgO-Lq thermometer of Helz & Thornber (1987) using Thermobar
T_C_MI = tb.calculate_liq_only_temp(liq_comps=MI_tb, equationT="T_Helz1987_MgO")-273.15

# adds temperatures to original dataframe
if "T_C" not in df_MI.columns:
    df_MI.insert(1,"T_C",T_C_MI)

# save your results
df_MI.to_csv("output/wieser2021_w_temperatures.csv")

## 1.1.5 Plotting

We can plot the MgO content against the temperature to see the correlation.

In [ ]:
# if you haven't run this Exercise, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#df_MI = pd.read_csv("answers/wieser2021_w_temperatures.csv")

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4,4))


ax1.plot(df_MI['MgO'], df_MI['T_C'], 'ok', mfc='blue')

ax1.set_xlabel('MgO [m] (wt%)')
ax1.set_ylabel('T (°C)')

We will use these results in future calculations of pressure and fluid composition!